In [2]:
# ==========================================
# 11. EXCEPTION HANDLING (Selected: Q126)
# ==========================================
# Question 126:
# Design a custom exception hierarchy for a payment processing system: PaymentError (base), 
# InsufficientFundsError, CardDeclinedError, InvalidCurrencyError, NetworkTimeoutError. Each should carry 
# relevant fields and a structured __str__. Write a process_payment() function that raises different 
# exceptions based on input parameters.
#
# Sample Input:  account_balance = 50.0, payment_amount = 100.0, currency = "USD"
# Sample Output: InsufficientFundsError: Requested 100.0 USD, but balance is only 50.0 USD.

class PaymentError(Exception):
    def __init__(self, message="Payment processing failed"):
        self.message = message
        super().__init__(self.message)

    def __str__(self):
        return f"{self.__class__.__name__}: {self.message}"

class InsufficientFundsError(PaymentError):
    def __init__(self, requested, balance, currency):
        self.requested = requested
        self.balance = balance
        self.currency = currency
        super().__init__(f"Requested {requested} {currency}, but balance is only {balance} {currency}.")

class CardDeclinedError(PaymentError):
    def __init__(self, card_last4, reason):
        self.card_last4 = card_last4
        self.reason = reason
        super().__init__(f"Card ending in {card_last4} was declined. Reason: {reason}")

class InvalidCurrencyError(PaymentError):
    def __init__(self, currency):
        self.currency = currency
        super().__init__(f"Currency '{currency}' is not supported for processing.")

class NetworkTimeoutError(PaymentError):
    def __init__(self, gateway):
        self.gateway = gateway
        super().__init__(f"Network timeout occurred while communicating with gateway '{gateway}'.")

def process_payment(amount, balance, currency="USD", card_last4="1234", simulate_network_fail=False):
        if amount > balance: raise InsufficientFundsError(amount,balance,currency)
        if not card_last4.isdigit(): raise CardDeclinedError(card_last4)
        if currency != "USD": raise InvalidCurrencyError(currency)
        if simulate_network_fail: raise NetworkTimeoutError("Payment Gateway")
        return "Payment successful"


if __name__ == '__main__':
    # Test Question 126
    try:
        process_payment(amount=100.0, balance=50.0, currency="USD")
    except PaymentError as e:
        print("Q126 Output:", str(e))

Q126 Output: InsufficientFundsError: Requested 100.0 USD, but balance is only 50.0 USD.


In [ ]:
#==========================================
# 11. EXCEPTION HANDLING (Selected: Q127)
# ==========================================
# Question 127:
# Write a context manager class ManagedDB using __enter__ and __exit__ that simulates a database 
# connection: it "opens" on enter (prints a message), "commits" on clean exit, and "rolls back" if an exception 
# occurs. Also implement it as a @contextmanager generator function and verify both produce identical 
# behaviour.
#
# Sample Input:  managed_db_class(fail=False) vs managed_db_gen(fail=True)
# Sample Output: ("Class: Committed", "Gen: Rolled back")
from contextlib import contextmanager
class ManagedDBClass:
    def __init__(self, db_name="test_db"):
        self.db_name = db_name

    def __enter__(self):
        print("Opening",self.db_name)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_tb is None:
            print("Committed")
        else:
            print("Roll back")
        return False

@contextmanager
def managed_db_gen(db_name="test_db"):
    print("Gen : Opening")
    try:
        yield
        print("Committed")
    except:
        print("Roll back")
        raise

if __name__ == '__main__':
    # Test Question 127
    res_class, res_gen = None, None
    
    # Class-based test
    with ManagedDBClass("test_db") as db:
        res_class = "Class: Committed"

    # Generator-based test (simulating rollback)
    try:
        with managed_db_gen("test_db"):
            raise ValueError("Simulated DB Failure")
    except ValueError:
        res_gen = "Gen: Rolled back"

    # print("Q127 Output:", (res_class, res_gen))



Opening test_db
Committed
Gen : Opening
Roll back
Q127 Output: ('Class: Committed', 'Gen: Rolled back')


In [11]:

# ==========================================
# 11. EXCEPTION HANDLING (Selected: Q128)
# ==========================================
# Question 128:
# Write a function safe_parse_all(data_list, parser_func) that attempts to parse every item with 
# parser_func, collects all (index, item, error) for failures, and returns (successes: list, failures: list[dict]). Do 
# not stop on the first error. Test with a list that has 30% intentionally malformed entries.
#
# Sample Input:  data_list = ["10", "20", "invalid", "30", "bad_num"], parser_func = int
# Sample Output: ([10, 20, 30], [{'index': 2, 'item': 'invalid', 'error': 'invalid literal for int()'}, {'index': 4, 'item': 'bad_num', 'error': 'invalid literal for int()'}])

def safe_parse_all(data_list, parser_func):
    successes = []
    failures = []
    try:
        for index,val in enumerate(data_list):
            v = parser_func(val)
            successes.append(v)
    except Exception as e:
        failures.append(
                    {'index':index , 'item': val, 'error': str(e)}
        )
    return successes,failures

if __name__ == '__main__':
    # Test Question 128
    raw_data = ["10", "20", "invalid", "30", "40", "bad_num", "50"]
    successes, failures = safe_parse_all(raw_data, int)
    print("Q128 Output:", (successes, failures))



Q128 Output: ([10, 20], [{'index': 2, 'item': 'invalid', 'error': "invalid literal for int() with base 10: 'invalid'"}])


In [14]:

# ==========================================
# 11. EXCEPTION HANDLING (Selected: Q129)
# ==========================================
# Question 129:
# Write a decorator @structured_log(logger_name) that wraps a function and, on any exception, logs 
# a structured dict {function, args, kwargs, exception_type, message, traceback} using Python's logging 
# module before re-raising. Demonstrate with three different exceptions.
#
# Sample Input:  func(a=1, b=0) raising ZeroDivisionError
# Sample Output: Caught ZeroDivisionError and re-raised successfully.
import logging
import traceback
import functools
def structured_log(logger_name="AppLogger"):
    logger = logging.getLogger(logger_name)
    
    def function(func):
        @functools.wraps(func)
        def wrapper(*args,**kwargs):
            try: return func(*args,**kwargs)
            except Exception as e:
                log_data = {
                    "function": func.__name__,
                    "args": args,
                    "kwargs": kwargs,
                    "exception_type": type(e).__name__,
                    "message": str(e),
                    "traceback": traceback.format_exc()
                }
                logger.error(log_data)
                raise
        return wrapper
    return function

@structured_log(logger_name="PaymentSystemLogger")
def divide_numbers(a, b):
    return a/b

if __name__ == '__main__':
    # Test Question 129
    try:
        divide_numbers(10, 0)
    except ZeroDivisionError as e:
        print("Q129 Output:", f"Caught {type(e).__name__} and re-raised successfully.")

{'function': 'divide_numbers', 'args': (10, 0), 'kwargs': {}, 'exception_type': 'ZeroDivisionError', 'message': 'division by zero', 'traceback': 'Traceback (most recent call last):\n  File "C:\\Users\\Sasiv\\AppData\\Local\\Temp\\ipykernel_20164\\1589470520.py", line 20, in wrapper\n    try: return func(*args,**kwargs)\n                ~~~~^^^^^^^^^^^^^^^^\n  File "C:\\Users\\Sasiv\\AppData\\Local\\Temp\\ipykernel_20164\\1589470520.py", line 37, in divide_numbers\n    return a/b\n           ~^~\nZeroDivisionError: division by zero\n'}


Q129 Output: Caught ZeroDivisionError and re-raised successfully.
